# Backtest with Prompt Optimization (2010-2023)

This notebook demonstrates Phase 1 of the workflow:
1.  **Backtest**: Running the agentic pipeline over historical data (2010-2023).
2.  **Training/Optimization**: Iteratively refining the instructions (prompts) for each sub-agent (Alpha, Risk, Portfolio) based on backtest performance.
3.  **Saving**: Persisting the optimized prompts for use in the out-of-sample test.

### Prerequisites
Ensure you have the environment variables set for `OPENAI_API_KEY`, `ALPACA_API_KEY`, and `ALPACA_SECRET_KEY`.

In [3]:
import os
import sys
import json
import pandas as pd
from datetime import datetime
from pathlib import Path

# Add project root to path to import Orchestrator
project_root = Path("../").resolve()
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "FinAgents" / "orchestrator_demo"))
sys.path.insert(0, str(project_root / "FinAgents" / "agent_pools"))

# Import Orchestrator
from FinAgents.orchestrator_demo.orchestrator import Orchestrator

# Initialize Orchestrator
orchestrator = Orchestrator()
print("✅ Orchestrator Initialized")

2026-02-01 02:31:46,600 - ExecutionAgent - INFO - Using Mock Alpaca Service (invalid or mock keys detected)


✅ Orchestrator Initialized


### Step 1: Define Optimization Loop

We simulate a training loop where we run a backtest for a specific year, evaluate performance, and ask a "Meta-Agent" to improve the instructions if targets aren't met.

In [4]:
def train_agents_over_period(symbol, start_year, end_year):
    current_prompts = {
        "Alpha": orchestrator.alpha_agent.agent.instructions,
        "Risk": orchestrator.risk_agent.agent.instructions,
        "Portfolio": orchestrator.portfolio_agent.agent.instructions
    }
    
    performance_history = []
    
    for year in range(start_year, end_year + 1):
        start_date = f"{year}-01-01"
        end_date = f"{year}-12-31"
        print(f"\n--- Processing Year: {year} ---")
        
        # Run Pipeline (using the Legacy pipeline method for direct control, or agentic if preferred)
        # Here we use the underlying run_pipeline logic exposed in Orchestrator
        # Note: In a real scenario, we would capture the result object
        try:
            result = orchestrator.run_pipeline(symbol, start_date, end_date, mode="backtest")
            
            if result and result.get('status') == 'success':
                metrics = result.get('performance_metrics', {})
                sharpe = metrics.get('sharpe_ratio', 0.0)
                print(f"📊 Performance for {year}: Sharpe Ratio = {sharpe:.2f}")
                
                performance_history.append({'year': year, 'sharpe': sharpe})
                
                # Optimization Logic: If performance is poor, optimize prompts
                if sharpe < 1.0: # Threshold for optimization
                    print("⚠️ Performance below threshold. Optimizing prompts...")
                    
                    # Call the optimizer (Meta-Agent)
                    # In the demo, this calls OpenAI to rewrite instructions
                    new_instruction = orchestrator.optimize_agent_prompts(
                        agent_name="Alpha", 
                        performance_metric="Sharpe Ratio", 
                        current_value=sharpe, 
                        target_value=1.5
                    )
                    
                    if new_instruction and "Optimization failed" not in new_instruction:
                         current_prompts["Alpha"] = new_instruction
                         print("✅ Alpha Agent prompt updated.")
            else:
                print(f"❌ Backtest failed for {year}: {result.get('message') if result else 'Unknown error'}")
                
        except Exception as e:
            print(f"❌ Error during execution: {e}")
            
    return current_prompts, performance_history

# Run the Training Loop
symbol = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'WMT']
optimized_prompts, history = train_agents_over_period(symbol, 2019, 2019)

2026-02-01 02:31:56,555 - Orchestrator - INFO - Running pipeline for AAPL, MSFT, GOOGL, AMZN, NVDA, META, TSLA, JPM, V, WMT from 2019-01-01 to 2019-12-31
2026-02-01 02:31:56,562 - Orchestrator - INFO - Fetching extended data from 2018-01-01 to 2019-12-31 for Rolling Training...
2026-02-01 02:31:56,566 - Orchestrator - INFO - Fetching data for ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'WMT'] from 2018-01-01 00:00:00 to 2019-12-31 00:00:00
2026-02-01 02:31:56,568 - Orchestrator - WARNING - yfinance not installed. Falling back to mock data.
2026-02-01 02:31:56,590 - Orchestrator - INFO - Data Split - Train: 2610 rows (2018-01-01 to 2019-01-01), Test: 2610 rows (2019-01-01 to 2019-12-31)
2026-02-01 02:31:56,593 - Orchestrator - ERROR - Pipeline error: AgentRunner.run_sync() cannot be called when an event loop is already running.



--- Processing Year: 2019 ---
DEBUG: 🤖 Requesting Alpha Agent LLM...
❌ Backtest failed for 2019: AgentRunner.run_sync() cannot be called when an event loop is already running.


Traceback (most recent call last):
  File "/Users/sangkyu/Work/tutorials/AgenticTrading/FinAgents/orchestrator_demo/orchestrator.py", line 331, in run_pipeline
    alpha_result = self.alpha_agent.generate_signals_from_data(
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sangkyu/Work/tutorials/AgenticTrading/FinAgents/agent_pools/alpha_agent_demo/alpha_signal_agent.py", line 498, in generate_signals_from_data
    result = Runner.run_sync(self.agent, request, context=context)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sangkyu/Work/tutorials/AgenticTrading/.venv/lib/python3.12/site-packages/agents/run.py", line 452, in run_sync
    return runner.run_sync(
           ^^^^^^^^^^^^^^^^
  File "/Users/sangkyu/Work/tutorials/AgenticTrading/.venv/lib/python3.12/site-packages/agents/run.py", line 867, in run_sync
    raise RuntimeError(
RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already runnin

### Step 2: Save Optimized Prompts

Save the evolved instructions to a file so they can be loaded for the out-of-sample test.

In [3]:
output_path = "optimized_prompts.json"
with open(output_path, "w") as f:
    json.dump(optimized_prompts, f, indent=2)
    
print(f"💾 Optimized prompts saved to {output_path}")
print("History:", history)

💾 Optimized prompts saved to optimized_prompts.json
History: []
